In [33]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

sparkConf = SparkConf()
sparkConf.setMaster("spark://spark-master:7077")
sparkConf.setAppName("GCSExample")
sparkConf.set("spark.driver.memory", "2g")
sparkConf.set("spark.executor.instances", "1")
sparkConf.set("spark.cores.max", "1")
sparkConf.set("spark.executor.cores", "1")
sparkConf.set("spark.driver.cores", "1")
# create the Spark session, which is the entry point to the Spark SQL engine.
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
print(0)
bucket = "assign2_temp"  # use your temp bucket 
spark.conf.set('temporaryGcsBucket', bucket)
# Setup Hadoop FS configuration for schema gs://
conf = spark.sparkContext._jsc.hadoopConfiguration()
conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
print(0)

conf = spark.sparkContext._jsc.hadoopConfiguration()

# Override Hadoop 3.5.0 GCS defaults
conf.setLong("fs.gs.block.size", 67108864)
conf.setLong("fs.gs.inputstream.inplace.seek.limit", 8388608)
conf.setInt("fs.gs.inputstream.min.range.request.size", 2097152)
conf.setInt("fs.gs.outputstream.buffer.size", 8388608)

# Make sure the correct GCS connector is used
conf.set(
    "fs.gs.impl",
    "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
)

conf.set(
    "fs.AbstractFileSystem.gs.impl",
    "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS"
)

print("GCS Hadoop configuration updated!")

#  Google Storage File Path
gsc_file_path = 'gs://assign2_data/steam_games.json'  
# Create data frame
df = spark.read.format("json").option("header", "true") \
       .load(gsc_file_path)
df.printSchema()
print(0)
newDf = df.select('genres')  # select one column

newDf.show()

df.show()
df.count()

0
0
GCS Hadoop configuration updated!
root
 |-- app_name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount_price: double (nullable = true)
 |-- early_access: boolean (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: string (nullable = true)
 |-- metascore: string (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- reviews_url: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- specs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)

0
+--------------------+
|              genres|
+--------------------+
|[Action, Casual, ...|
|[Free to Play, In...|
|[Casual, Free to ...|
|[Action, Adventur...|
|               

32135

how many distinct products have been sold in each store? 


In [31]:
dfw = df.select("genres","publisher").distinct()  #Returns a new DataFrame containing the distinct rows in this DataFrame.
dfw.show(20)
dfw.count()

+--------------------+--------------------+
|              genres|           publisher|
+--------------------+--------------------+
|            [Action]|              Capcom|
|     [Action, Indie]|        RichMakeGame|
|    [Action, Racing]|Remedy Entertainm...|
|  [Action, Strategy]|Warner Bros. Inte...|
|[Adventure, Free ...|       Crystal Shard|
|         [Adventure]|      Microids Indie|
| [Action, Adventure]|BANDAI NAMCO Ente...|
|[Action, Free to ...|                NULL|
|[Casual, Indie, S...|                NULL|
|[Action, Indie, S...|         David Stark|
|     [Casual, Indie]|      Strategy First|
|[Simulation, Stra...|             Ubisoft|
|[Adventure, Indie...|     Loudpak Studios|
|     [Action, Indie]|              TERNOX|
|[Indie, Simulatio...|        Texel Raptor|
|[Action, Adventur...|          Red Dahlia|
| [Free to Play, RPG]|         Kiz Studios|
|   [Indie, Strategy]|    Turbo Tape Games|
|[Action, Indie, R...|              2xMilk|
|[Design &amp; Ill...| MAGIX Sof

12813

In [3]:
# Example dataframe write
dfw.write \
    .mode("overwrite") \
    .csv("gs://de2026_data/lab75out.csv") # use correct bucket name

In [4]:
# Stop the Spark context
spark.stop()